**AUTHOR: ANTOINE AMIEL**

In [14]:
# Import necessary libraries
import numpy as np
import sys
sys.path.append('../utils/')
sys.path.append('../src/')

from initial_param_SPA import initialize_parameters
from SPA import SPA
from plot_RESULT import plot_RESULT
from skimage.metrics import peak_signal_noise_ratio as psnr

In [ ]:
# Hyperparameters

N_MC=300 # Total number of MCMC iterations. By default take 1000.
N_burn_in=200 # Number of burn-in iterations. By default take 200.
rho=20 # User-defined standard deviation of the variable of interest x. By default take 20.
alpha=1 # User-defined hyperparameter of the prior p(u). By default take 1.
KERNEL_SIZE=39 # Size of the Kernel filter (size x size).
KERNEL_SIGMA=4 # Standard deviation of the kernel Gaussian
PATH_IMAGE='lena.bmp' # path of the image x
GAMMA = 6e-3 # Regularization parameter
DELTA = 1e-1 # Regularization parameter for the Laplacian 
TARGET_SNR=30 # Target SNR in dB
SEED=1 # SEED for reproductibility

In [ ]:
# Load all parameters
params = initialize_parameters(kernel_size=KERNEL_SIZE, kernel_sigma=KERNEL_SIGMA, 
                               path_image=PATH_IMAGE, gamma=GAMMA, delta = DELTA, TARGET=TARGET_SNR, seed=SEED)

# Unpack parameters
D = params['D']
mu1 = params['mu1']
F_blur_kernel = params['F_blur_kernel']
F_Laplace = params['F_Laplace'] # smooth prior
img_noisy = params['img_noisy']
gamma = params['gamma']
N = params['N']
N_bi = params['N_burn_in']
img_original = params['img_original']

Initial parameters loaded!


## Load workspace variables and launch SPA algorithm
Load workspace variables (defined in `../utils/initial_param_SPA.py`) and launch SPA algorithm.

In [17]:
# Run SPA algorithm
X_MC, _, _ = SPA(D, mu1, F_blur_kernel, rho, alpha, img_noisy, gamma, F_Laplace, N, N_MC)


BEGINNING OF THE SAMPLING


Sampling in progress: 100%|██████████| 299/299 [04:13<00:00,  1.18it/s]


END OF THE GIBBS SAMPLING
Execution time of the Gibbs sampling: 262.67 sec


## Display PSNR and SNR
Display PSNR and SNR associated to the MMSE estimator of x.

In [18]:
# Calculate MMSE estimator (mean of samples after burn-in)
X_MMSE = np.mean(X_MC[:, :, N_bi:N_MC], axis=2)

# Calculate PSNR and SNR
PSNR = psnr(img_original.astype(np.uint8), X_MMSE.astype(np.uint8))

# Calculate SNR manually (since skimage doesn't have SNR)
signal_power = np.sum(img_original**2)
noise_power = np.sum((img_original - X_MMSE)**2)
SNR = 10 * np.log10(signal_power / noise_power)

print(f'PSNR: {PSNR:.4f} dB')
print(f'SNR: {SNR:.4f} dB')

PSNR: 20.7204 dB
SNR: 19.7177 dB


## Plot the results
Display the original image, degraded image, and reconstructed results.

In [ ]:
# Plot results
plot_RESULT(img_noisy, img_original, X_MC, N_bi, N)